# London Intelligence Dataset — Integration

## Objective

This notebook is responsible for integrating the independently validated data layers of the London Property Intelligence project into a single analytical dataset at the:

**Borough × Year**

level.

The integration combines the validated feature layers while preserving the original data grain, data provenance, and temporal coverage of each source.

## Independent Feature Layers

The current project layers include:

- Property / Transactions
- Population
- Crime
- Income
- PTAL
- IMD

Each layer is maintained independently in `data/processed/` and is treated as a validated source before integration.

## Integration Principles

- Original independent feature layers remain unchanged during integration.
- Each layer is inspected and validated before integration.
- The grain of each dataset is explicitly verified before merging.
- Merge keys are defined explicitly and used consistently.
- `Borough × Year` uniqueness is checked before and after each merge.
- Missing values are preserved when a source layer does not cover a particular year.
- Artificial values and unexplained extrapolation are avoided.
- The final integrated dataset is validated before feature engineering and modelling.

## Integration Workflow

The integration process will follow these stages:

1. Inspect the existing processed datasets.
2. Identify the appropriate base dataset.
3. Verify the grain and temporal coverage of each layer.
4. Verify the availability and consistency of merge keys.
5. Integrate the validated feature layers step by step.
6. Validate the dataset after each integration step.
7. Perform a final audit of the integrated dataset.
8. Export the final integrated dataset only after all validation checks pass.

## Current Phase

The current phase is **inspection and validation**.

Before performing any merge, the existing processed datasets will be inspected to determine their structure, grain, temporal coverage, and role in the integration process.

No merge or final export will be performed until the structure and grain of the required layers have been confirmed.

## Expected Final Outcome

The final output will be a validated London Intelligence Dataset at the:

**Borough × Year**

level, combining the relevant property and urban intelligence features available for each borough and year.

The resulting dataset will serve as the foundation for:

- Exploratory Data Analysis (EDA)
- Feature Engineering
- Predictive Modelling
- Investment Analysis
- Streamlit Dashboard Development

In [1]:
import pandas as pd

In [2]:
# Inspect borough_year_features.csv

borough_year_path = "../data/processed/borough_year_features.csv"

borough_year_features = pd.read_csv(borough_year_path)

print("=== BOROUGH YEAR FEATURES — BASIC INSPECTION ===")

print("Shape:", borough_year_features.shape)

print("\nColumns:")
for i, column in enumerate(borough_year_features.columns, start=1):
    print(f"{i}. {column}")

print("\nData types:")
print(borough_year_features.dtypes)

=== BOROUGH YEAR FEATURES — BASIC INSPECTION ===
Shape: (264, 12)

Columns:
1. District
2. Year
3. Transactions
4. Average_Price
5. Median_Price
6. Min_Price
7. Max_Price
8. Price_STD
9. Average_Price_Growth
10. Median_Price_Growth
11. Target_Average_Price_Growth
12. Target_Median_Price_Growth

Data types:
District                        object
Year                             int64
Transactions                     int64
Average_Price                  float64
Median_Price                   float64
Min_Price                        int64
Max_Price                        int64
Price_STD                      float64
Average_Price_Growth           float64
Median_Price_Growth            float64
Target_Average_Price_Growth    float64
Target_Median_Price_Growth     float64
dtype: object


## Inspect Property Time Coverage and Grain

The `borough_year_features` dataset contains 264 observations across 33 London boroughs and multiple years.

Before using it as a potential integration base, its borough coverage, year coverage, and `Borough × Year` uniqueness are verified.

In [3]:
print("=== PROPERTY DATASET — GRAIN & COVERAGE ===")

print("Unique districts:", borough_year_features["District"].nunique())

print("\nYears:")
print(sorted(borough_year_features["Year"].unique()))

print("\nNumber of years:", borough_year_features["Year"].nunique())

print("\nDuplicate District-Year records:",
      borough_year_features.duplicated(
          subset=["District", "Year"]
      ).sum())

print("\nRows per district:")
print(
    borough_year_features
    .groupby("District")["Year"]
    .nunique()
    .value_counts()
    .sort_index()
)

=== PROPERTY DATASET — GRAIN & COVERAGE ===
Unique districts: 33

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Number of years: 8

Duplicate District-Year records: 0

Rows per district:
Year
8    33
Name: count, dtype: int64


## Compare Geographic Identifiers Across Feature Layers

Before integration, the geographic identifiers used by the independent feature layers are compared.

This comparison is performed before any merge to identify naming differences, missing boroughs, or inconsistent geographic keys.

No source dataset is modified during this inspection.

In [4]:
# Inspect geographic identifiers and temporal coverage across processed feature layers

files_to_inspect = {
    "Property": "../data/processed/borough_year_features.csv",
    "Population": "../data/processed/london_population.csv",
    "Crime": "../data/processed/london_crime.csv",
    "Income": "../data/processed/london_income.csv",
    "PTAL": "../data/processed/borough_ptai_2015.csv",
    "IMD": "../data/processed/london_imd.csv",
}

datasets = {}

for layer, path in files_to_inspect.items():
    df = pd.read_csv(path)
    datasets[layer] = df

    print(f"\n{'=' * 70}")
    print(f"{layer}")
    print(f"{'=' * 70}")

    print("Shape:", df.shape)

    print("Columns:")
    print(df.columns.tolist())

    print("Year column:", "Year" if "Year" in df.columns else "Not present")

    if "Year" in df.columns:
        print("Year coverage:", sorted(df["Year"].dropna().unique().tolist()))

    print("Potential geographic columns:")
    geographic_candidates = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in ["borough", "district", "area", "name", "code", "bocu", "geography"])
    ]
    print(geographic_candidates)


Property
Shape: (264, 12)
Columns:
['District', 'Year', 'Transactions', 'Average_Price', 'Median_Price', 'Min_Price', 'Max_Price', 'Price_STD', 'Average_Price_Growth', 'Median_Price_Growth', 'Target_Average_Price_Growth', 'Target_Median_Price_Growth']
Year column: Year
Year coverage: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Potential geographic columns:
['District']

Population
Shape: (396, 5)
Columns:
['Code', 'Name', 'Geography', 'Year', 'Population']
Year column: Year
Year coverage: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]
Potential geographic columns:
['Code', 'Name', 'Geography']

Crime
Shape: (96, 15)
Columns:
['BOCU', 'Year', 'ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES', 'FRAUD AND FORGERY', 'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS', 'PUBLIC ORDER OFFENCES', 'ROBBERY', 'SEXUAL OFFENCES', 'THEFT', 'VEHICLE OFFENCES', 'VIOLENCE AGAINST THE PERSON', 'Merge_Key']
Year column: Year
Year coverage: [2021, 2022, 

## Inspect Transaction Source Datasets

The transaction datasets are inspected separately to clarify their role in the project architecture.

These files are upstream transaction sources and are not assumed to be integration layers at this stage.

In [5]:
transaction_files = {
    "Transactions": "../data/processed/london_transactions.csv",
    "Transactions 2018–2025": "../data/processed/london_transactions_2018_2025.csv",
}

for name, path in transaction_files.items():
    df = pd.read_csv(path)

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "Year" in df.columns:
        print("Year coverage:", sorted(df["Year"].dropna().unique().tolist()))


Transactions
Shape: (100841, 16)
Columns: ['Transaction unique identifier', 'Price', 'Date of Transfer', 'Postcode', 'Property Type', 'Old/New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town/City', 'District', 'County', 'PPD Category Type', 'Record Status']

Transactions 2018–2025
Shape: (887229, 17)
Columns: ['Transaction unique identifier', 'Price', 'Date of Transfer', 'Postcode', 'Property Type', 'Old/New', 'Duration', 'PAON', 'SAON', 'Street', 'Locality', 'Town/City', 'District', 'County', 'PPD Category Type', 'Record Status', 'Year']
Year coverage: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Feature Layer Coverage Audit

The independent feature layers are audited before integration to establish their geographic coverage, temporal coverage, and data grain.

This audit is used to determine how each layer can be integrated into the `Borough × Year` analytical dataset.

No source dataset is modified during this process.

In [6]:
coverage_audit = []

for layer, df in datasets.items():
    year_values = (
        sorted(df["Year"].dropna().unique().tolist())
        if "Year" in df.columns
        else []
    )

    coverage_audit.append({
        "Layer": layer,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Year_Min": min(year_values) if year_values else None,
        "Year_Max": max(year_values) if year_values else None,
        "Number_of_Years": len(year_values),
    })

coverage_audit_df = pd.DataFrame(coverage_audit)

coverage_audit_df

,Layer,Rows,Columns,Year_Min,Year_Max,Number_of_Years
0,Property,264,12,2018.0,2025.0,8
1,Population,396,5,2011.0,2022.0,12
2,Crime,96,15,2021.0,2023.0,3
3,Income,198,8,2018.0,2023.0,6
4,PTAL,33,4,NaN,NaN,0
5,IMD,33,13,2019.0,2019.0,1


## Validate Feature Layer Grain

Each independent feature layer is validated against its expected analytical grain before integration.

Time-varying layers are expected to contain one record per `Borough × Year`, while static borough-level layers are expected to contain one record per `Borough`.

No source dataset is modified during validation.

In [7]:
grain_audit = {}

# Property
grain_audit["Property"] = {
    "Expected Grain": "District × Year",
    "Rows": len(datasets["Property"]),
    "Unique District-Year": datasets["Property"]
        .drop_duplicates(["District", "Year"])
        .shape[0],
    "Duplicate District-Year": datasets["Property"]
        .duplicated(["District", "Year"])
        .sum()
}

# Population
grain_audit["Population"] = {
    "Expected Grain": "Code × Year",
    "Rows": len(datasets["Population"]),
    "Unique Code-Year": datasets["Population"]
        .drop_duplicates(["Code", "Year"])
        .shape[0],
    "Duplicate Code-Year": datasets["Population"]
        .duplicated(["Code", "Year"])
        .sum()
}

# Crime
grain_audit["Crime"] = {
    "Expected Grain": "BOCU × Year",
    "Rows": len(datasets["Crime"]),
    "Unique BOCU-Year": datasets["Crime"]
        .drop_duplicates(["BOCU", "Year"])
        .shape[0],
    "Duplicate BOCU-Year": datasets["Crime"]
        .duplicated(["BOCU", "Year"])
        .sum()
}

# Income
grain_audit["Income"] = {
    "Expected Grain": "Merge_Key × Year",
    "Rows": len(datasets["Income"]),
    "Unique Merge_Key-Year": datasets["Income"]
        .drop_duplicates(["Merge_Key", "Year"])
        .shape[0],
    "Duplicate Merge_Key-Year": datasets["Income"]
        .duplicated(["Merge_Key", "Year"])
        .sum()
}

# PTAL
grain_audit["PTAL"] = {
    "Expected Grain": "Borough",
    "Rows": len(datasets["PTAL"]),
    "Unique Borough": datasets["PTAL"]
        .drop_duplicates(["Borough Name"])
        .shape[0],
    "Duplicate Borough": datasets["PTAL"]
        .duplicated(["Borough Name"])
        .sum()
}

# IMD
grain_audit["IMD"] = {
    "Expected Grain": "Borough",
    "Rows": len(datasets["IMD"]),
    "Unique Borough": datasets["IMD"]
        .drop_duplicates(["Borough_Code"])
        .shape[0],
    "Duplicate Borough": datasets["IMD"]
        .duplicated(["Borough_Code"])
        .sum()
}

grain_audit_df = pd.DataFrame(grain_audit).T

grain_audit_df

,Expected Grain,Rows,Unique District-Year,Duplicate District-Year,Unique Code-Year,Duplicate Code-Year,Unique BOCU-Year,Duplicate BOCU-Year,Unique Merge_Key-Year,Duplicate Merge_Key-Year,Unique Borough,Duplicate Borough
Property,District × Year,264,264,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Population,Code × Year,396,NaN,NaN,396,0,NaN,NaN,NaN,NaN,NaN,NaN
Crime,BOCU × Year,96,NaN,NaN,NaN,NaN,96,0,NaN,NaN,NaN,NaN
Income,Merge_Key × Year,198,NaN,NaN,NaN,NaN,NaN,NaN,198,0,NaN,NaN
PTAL,Borough,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33,0
IMD,Borough,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33,0


## Geographic Identifier Audit

The geographic identifiers used by each feature layer are inspected before integration.

The purpose is to establish a consistent borough-level mapping without modifying any source dataset.

No merge is performed at this stage.

In [8]:
print("=" * 80)
print("GEOGRAPHIC IDENTIFIER AUDIT")
print("=" * 80)

identifier_columns = {
    "Property": ["District"],
    "Population": ["Code", "Name", "Geography"],
    "Crime": ["BOCU", "Merge_Key"],
    "Income": ["Code", "Area", "Merge_Key"],
    "PTAL": ["Borough Code", "Borough Name"],
    "IMD": ["Borough_Code", "Borough_Name"],
}

for layer, columns in identifier_columns.items():
    df = datasets[layer]

    print(f"\n{'=' * 80}")
    print(layer)
    print(f"{'=' * 80}")

    available_columns = [col for col in columns if col in df.columns]

    for column in available_columns:
        print(f"\n{column}")
        print("Unique values:", df[column].nunique(dropna=True))
        print("Sample:")
        print(df[column].drop_duplicates().head(10).tolist())

GEOGRAPHIC IDENTIFIER AUDIT

Property

District
Unique values: 33
Sample:
['BARKING AND DAGENHAM', 'BARNET', 'BEXLEY', 'BRENT', 'BROMLEY', 'CAMDEN', 'CITY OF LONDON', 'CITY OF WESTMINSTER', 'CROYDON', 'EALING']

Population

Code
Unique values: 33
Sample:
['E09000007', 'E09000001', 'E09000012', 'E09000013', 'E09000014', 'E09000019', 'E09000020', 'E09000022', 'E09000023', 'E09000025']

Name
Unique values: 33
Sample:
['Camden', 'City of London', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Islington', 'Kensington and Chelsea', 'Lambeth', 'Lewisham', 'Newham']

Geography
Unique values: 1
Sample:
['London Borough']

Crime

BOCU
Unique values: 32
Sample:
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich']

Merge_Key
Unique values: 32
Sample:
['BARKING AND DAGENHAM', 'BARNET', 'BEXLEY', 'BRENT', 'BROMLEY', 'CAMDEN', 'CROYDON', 'EALING', 'ENFIELD', 'GREENWICH']

Income

Code
Unique values: 33
Sample:
['E09000001', 'E09000

In [9]:
# Geographic identifier counts

identifier_columns = {
    "Property": "District",
    "Population": "Name",
    "Crime": "Merge_Key",
    "Income": "Area",
    "PTAL": "Borough Name",
    "IMD": "Borough_Name",
}

print("=== GEOGRAPHIC IDENTIFIER COUNTS ===")

for layer, column in identifier_columns.items():

    df = datasets[layer]

    print(
        f"{layer:<12} | "
        f"Column: {column:<20} | "
        f"Unique: {df[column].nunique(dropna=True):>2} | "
        f"Missing: {df[column].isna().sum():>2}"
    )

=== GEOGRAPHIC IDENTIFIER COUNTS ===
Property     | Column: District             | Unique: 33 | Missing:  0
Population   | Column: Name                 | Unique: 33 | Missing:  0
Crime        | Column: Merge_Key            | Unique: 32 | Missing:  0
Income       | Column: Area                 | Unique: 33 | Missing:  0
PTAL         | Column: Borough Name         | Unique: 33 | Missing:  0
IMD          | Column: Borough_Name         | Unique: 33 | Missing:  0


In [10]:
# Inspect Crime geographic identifiers

crime = datasets["Crime"]

print("=== CRIME GEOGRAPHIC IDENTIFIER AUDIT ===")

print("Unique Merge_Key values:", crime["Merge_Key"].nunique())
print("Unique BOCU values:", crime["BOCU"].nunique())

print("\nMerge_Key values:")
for value in sorted(crime["Merge_Key"].unique()):
    print(value)

=== CRIME GEOGRAPHIC IDENTIFIER AUDIT ===
Unique Merge_Key values: 32
Unique BOCU values: 32

Merge_Key values:
BARKING AND DAGENHAM
BARNET
BEXLEY
BRENT
BROMLEY
CAMDEN
CITY OF WESTMINSTER
CROYDON
EALING
ENFIELD
GREENWICH
HACKNEY
HAMMERSMITH AND FULHAM
HARINGEY
HARROW
HAVERING
HILLINGDON
HOUNSLOW
ISLINGTON
KENSINGTON AND CHELSEA
KINGSTON UPON THAMES
LAMBETH
LEWISHAM
MERTON
NEWHAM
REDBRIDGE
RICHMOND UPON THAMES
SOUTHWARK
SUTTON
TOWER HAMLETS
WALTHAM FOREST
WANDSWORTH


In [11]:
# Identify the borough missing from Crime

property_boroughs = set(
    datasets["Property"]["District"]
    .dropna()
    .str.strip()
    .str.upper()
    .unique()
)

crime_boroughs = set(
    datasets["Crime"]["Merge_Key"]
    .dropna()
    .str.strip()
    .str.upper()
    .unique()
)

missing_from_crime = sorted(
    property_boroughs - crime_boroughs
)

extra_in_crime = sorted(
    crime_boroughs - property_boroughs
)

print("=== CRIME BOROUGH COVERAGE CHECK ===")

print("Property boroughs:", len(property_boroughs))
print("Crime boroughs:", len(crime_boroughs))

print("\nMissing from Crime:")
print(missing_from_crime)

print("\nExtra in Crime:")
print(extra_in_crime)

=== CRIME BOROUGH COVERAGE CHECK ===
Property boroughs: 33
Crime boroughs: 32

Missing from Crime:
['CITY OF LONDON']

Extra in Crime:
[]


In [12]:
# Compare borough codes across feature layers

code_columns = {
    "Population": "Code",
    "PTAL": "Borough Code",
    "IMD": "Borough_Code",
}

print("=== BOROUGH CODE COVERAGE ===")

for layer, column in code_columns.items():

    df = datasets[layer]

    codes = (
        df[column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    print(
        f"{layer:<12} | "
        f"Column: {column:<15} | "
        f"Unique codes: {len(codes):>2}"
    )

=== BOROUGH CODE COVERAGE ===
Population   | Column: Code            | Unique codes: 33
PTAL         | Column: Borough Code    | Unique codes: 33
IMD          | Column: Borough_Code    | Unique codes: 33


In [13]:
# Compare borough code sets

population_codes = set(
    datasets["Population"]["Code"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

ptal_codes = set(
    datasets["PTAL"]["Borough Code"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

imd_codes = set(
    datasets["IMD"]["Borough_Code"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("=== BOROUGH CODE SET COMPARISON ===")

print("Population codes:", len(population_codes))
print("PTAL codes:", len(ptal_codes))
print("IMD codes:", len(imd_codes))

print("\nPopulation vs PTAL — differences:")
print("Only in Population:", sorted(population_codes - ptal_codes))
print("Only in PTAL:", sorted(ptal_codes - population_codes))

print("\nPopulation vs IMD — differences:")
print("Only in Population:", sorted(population_codes - imd_codes))
print("Only in IMD:", sorted(imd_codes - population_codes))

print("\nPTAL vs IMD — differences:")
print("Only in PTAL:", sorted(ptal_codes - imd_codes))
print("Only in IMD:", sorted(imd_codes - ptal_codes))

=== BOROUGH CODE SET COMPARISON ===
Population codes: 33
PTAL codes: 33
IMD codes: 33

Population vs PTAL — differences:
Only in Population: []
Only in PTAL: []

Population vs IMD — differences:
Only in Population: []
Only in IMD: []

PTAL vs IMD — differences:
Only in PTAL: []
Only in IMD: []


In [14]:
# Build canonical Borough Code–Name reference from Population

population_reference = (
    datasets["Population"][
        ["Code", "Name"]
    ]
    .drop_duplicates()
    .sort_values("Code")
    .reset_index(drop=True)
)

print("=== CANONICAL BOROUGH REFERENCE ===")
print("Rows:", len(population_reference))
print("Unique codes:", population_reference["Code"].nunique())
print("Unique names:", population_reference["Name"].nunique())

display(population_reference)

=== CANONICAL BOROUGH REFERENCE ===
Rows: 33
Unique codes: 33
Unique names: 33


,Code,Name
0,E09000001,City of London
1,E09000002,Barking and Dagenham
2,E09000003,Barnet
3,E09000004,Bexley
4,E09000005,Brent
5,E09000006,Bromley
6,E09000007,Camden
7,E09000008,Croydon
8,E09000009,Ealing
9,E09000010,Enfield


In [15]:
# Audit Borough Name mapping against canonical reference

canonical_names = set(
    population_reference["Name"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

name_columns = {
    "Property": "District",
    "Crime": "Merge_Key",
    "Income": "Area",
    "PTAL": "Borough Name",
    "IMD": "Borough_Name",
}

print("=== BOROUGH NAME MAPPING AUDIT ===")

for layer, column in name_columns.items():

    names = set(
        datasets[layer][column]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    missing_from_reference = sorted(names - canonical_names)
    missing_from_layer = sorted(canonical_names - names)

    print(f"\n{layer}")
    print("-" * 60)
    print("Unique names:", len(names))
    print("Not found in canonical reference:", missing_from_reference)
    print("Missing from layer:", missing_from_layer)

=== BOROUGH NAME MAPPING AUDIT ===

Property
------------------------------------------------------------
Unique names: 33
Not found in canonical reference: ['BARKING AND DAGENHAM', 'BARNET', 'BEXLEY', 'BRENT', 'BROMLEY', 'CAMDEN', 'CITY OF LONDON', 'CITY OF WESTMINSTER', 'CROYDON', 'EALING', 'ENFIELD', 'GREENWICH', 'HACKNEY', 'HAMMERSMITH AND FULHAM', 'HARINGEY', 'HARROW', 'HAVERING', 'HILLINGDON', 'HOUNSLOW', 'ISLINGTON', 'KENSINGTON AND CHELSEA', 'KINGSTON UPON THAMES', 'LAMBETH', 'LEWISHAM', 'MERTON', 'NEWHAM', 'REDBRIDGE', 'RICHMOND UPON THAMES', 'SOUTHWARK', 'SUTTON', 'TOWER HAMLETS', 'WALTHAM FOREST', 'WANDSWORTH']
Missing from layer: ['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridg

In [16]:
# Define a controlled normalization function for borough matching

def normalize_borough_name(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace("-", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )

In [17]:
# Create normalized canonical borough reference

canonical_reference = population_reference.copy()

canonical_reference["Borough_Match_Key"] = normalize_borough_name(
    canonical_reference["Name"]
)

print("=== NORMALIZED CANONICAL REFERENCE AUDIT ===")

print("Rows:", len(canonical_reference))
print(
    "Unique match keys:",
    canonical_reference["Borough_Match_Key"].nunique()
)

print(
    "Duplicate match keys:",
    canonical_reference["Borough_Match_Key"]
    .duplicated()
    .sum()
)

display(canonical_reference)

=== NORMALIZED CANONICAL REFERENCE AUDIT ===
Rows: 33
Unique match keys: 33
Duplicate match keys: 0


,Code,Name,Borough_Match_Key
0,E09000001,City of London,CITY OF LONDON
1,E09000002,Barking and Dagenham,BARKING AND DAGENHAM
2,E09000003,Barnet,BARNET
3,E09000004,Bexley,BEXLEY
4,E09000005,Brent,BRENT
5,E09000006,Bromley,BROMLEY
6,E09000007,Camden,CAMDEN
7,E09000008,Croydon,CROYDON
8,E09000009,Ealing,EALING
9,E09000010,Enfield,ENFIELD


In [18]:
# Audit normalized borough mapping across all feature layers

print("=== NORMALIZED BOROUGH MAPPING AUDIT ===")

for layer, column in name_columns.items():

    df = datasets[layer]

    layer_keys = set(
        normalize_borough_name(df[column])
        .dropna()
        .unique()
    )

    canonical_keys = set(
        canonical_reference["Borough_Match_Key"]
    )

    missing_from_reference = sorted(
        layer_keys - canonical_keys
    )

    missing_from_layer = sorted(
        canonical_keys - layer_keys
    )

    print(f"\n{layer}")
    print("-" * 60)
    print("Unique normalized keys:", len(layer_keys))
    print(
        "Not found in canonical reference:",
        missing_from_reference
    )
    print(
        "Missing from layer:",
        missing_from_layer
    )

=== NORMALIZED BOROUGH MAPPING AUDIT ===

Property
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: ['CITY OF WESTMINSTER']
Missing from layer: ['WESTMINSTER']

Crime
------------------------------------------------------------
Unique normalized keys: 32
Not found in canonical reference: ['CITY OF WESTMINSTER']
Missing from layer: ['CITY OF LONDON', 'WESTMINSTER']

Income
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []

PTAL
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []

IMD
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []


In [19]:
# Define explicit borough name aliases for integration matching

borough_aliases = {
    "CITY OF WESTMINSTER": "WESTMINSTER"
}

def normalize_borough_for_integration(series):
    normalized = normalize_borough_name(series)

    return normalized.replace(borough_aliases)

In [20]:
# Re-run normalized borough mapping audit with explicit aliases

print("=== FINAL BOROUGH MAPPING AUDIT ===")

for layer, column in name_columns.items():

    df = datasets[layer]

    layer_keys = set(
        normalize_borough_for_integration(df[column])
        .dropna()
        .unique()
    )

    canonical_keys = set(
        canonical_reference["Borough_Match_Key"]
    )

    missing_from_reference = sorted(
        layer_keys - canonical_keys
    )

    missing_from_layer = sorted(
        canonical_keys - layer_keys
    )

    print(f"\n{layer}")
    print("-" * 60)
    print("Unique normalized keys:", len(layer_keys))
    print(
        "Not found in canonical reference:",
        missing_from_reference
    )
    print(
        "Missing from layer:",
        missing_from_layer
    )

=== FINAL BOROUGH MAPPING AUDIT ===

Property
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []

Crime
------------------------------------------------------------
Unique normalized keys: 32
Not found in canonical reference: []
Missing from layer: ['CITY OF LONDON']

Income
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []

PTAL
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []

IMD
------------------------------------------------------------
Unique normalized keys: 33
Not found in canonical reference: []
Missing from layer: []


In [21]:
# Inspect geographic codes across processed feature layers

code_columns = {
    "Population": "Code",
    "PTAL": "Borough Code",
    "IMD": "Borough_Code",
    "Income": "Code",
    "Crime": "BOCU",
}

print("=== GEOGRAPHIC CODE INSPECTION ===")

for layer, column in code_columns.items():

    df = datasets[layer]

    print("\n" + "=" * 70)
    print(layer)
    print("=" * 70)

    print("Column:", column)
    print("Data type:", df[column].dtype)
    print("Unique values:", df[column].nunique())
    print("Missing values:", df[column].isna().sum())

    print("\nFirst 10 unique values:")
    for i, value in enumerate(
        sorted(df[column].dropna().astype(str).unique())[:10],
        start=1
    ):
        print(f"{i:02d}. {value}")

=== GEOGRAPHIC CODE INSPECTION ===

Population
Column: Code
Data type: object
Unique values: 33
Missing values: 0

First 10 unique values:
01. E09000001
02. E09000002
03. E09000003
04. E09000004
05. E09000005
06. E09000006
07. E09000007
08. E09000008
09. E09000009
10. E09000010

PTAL
Column: Borough Code
Data type: object
Unique values: 33
Missing values: 0

First 10 unique values:
01. E09000001
02. E09000002
03. E09000003
04. E09000004
05. E09000005
06. E09000006
07. E09000007
08. E09000008
09. E09000009
10. E09000010

IMD
Column: Borough_Code
Data type: object
Unique values: 33
Missing values: 0

First 10 unique values:
01. E09000001
02. E09000002
03. E09000003
04. E09000004
05. E09000005
06. E09000006
07. E09000007
08. E09000008
09. E09000009
10. E09000010

Income
Column: Code
Data type: object
Unique values: 33
Missing values: 0

First 10 unique values:
01. E09000001
02. E09000002
03. E09000003
04. E09000004
05. E09000005
06. E09000006
07. E09000007
08. E09000008
09. E09000009
10. 

In [22]:
# Compare geographic codes against the canonical borough reference

canonical_codes = set(
    canonical_reference["Code"].astype(str)
)

print("=== CANONICAL CODE COMPATIBILITY AUDIT ===")

for layer, column in code_columns.items():

    df = datasets[layer]

    layer_codes = set(
        df[column]
        .dropna()
        .astype(str)
        .unique()
    )

    common_codes = layer_codes & canonical_codes
    only_in_layer = layer_codes - canonical_codes
    only_in_canonical = canonical_codes - layer_codes

    print("\n" + "=" * 70)
    print(layer)
    print("=" * 70)

    print("Column:", column)
    print("Unique codes:", len(layer_codes))
    print("Matching canonical codes:", len(common_codes))
    print("Codes only in layer:", sorted(only_in_layer))
    print("Canonical codes missing from layer:", sorted(only_in_canonical))

=== CANONICAL CODE COMPATIBILITY AUDIT ===

Population
Column: Code
Unique codes: 33
Matching canonical codes: 33
Codes only in layer: []
Canonical codes missing from layer: []

PTAL
Column: Borough Code
Unique codes: 33
Matching canonical codes: 33
Codes only in layer: []
Canonical codes missing from layer: []

IMD
Column: Borough_Code
Unique codes: 33
Matching canonical codes: 33
Codes only in layer: []
Canonical codes missing from layer: []

Income
Column: Code
Unique codes: 33
Matching canonical codes: 33
Codes only in layer: []
Canonical codes missing from layer: []

Crime
Column: BOCU
Unique codes: 32
Matching canonical codes: 0
Codes only in layer: ['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 

In [23]:
# Inspect Income and Crime identifiers in full

for layer, column in {
    "Income": "Code",
    "Crime": "BOCU"
}.items():

    df = datasets[layer]

    print("\n" + "=" * 70)
    print(layer)
    print("=" * 70)

    values = sorted(
        df[column]
        .dropna()
        .astype(str)
        .unique()
    )

    print("Column:", column)
    print("Unique values:", len(values))
    print("Values:")

    for i, value in enumerate(values, start=1):
        print(f"{i:02d}. {value}")


Income
Column: Code
Unique values: 33
Values:
01. E09000001
02. E09000002
03. E09000003
04. E09000004
05. E09000005
06. E09000006
07. E09000007
08. E09000008
09. E09000009
10. E09000010
11. E09000011
12. E09000012
13. E09000013
14. E09000014
15. E09000015
16. E09000016
17. E09000017
18. E09000018
19. E09000019
20. E09000020
21. E09000021
22. E09000022
23. E09000023
24. E09000024
25. E09000025
26. E09000026
27. E09000027
28. E09000028
29. E09000029
30. E09000030
31. E09000031
32. E09000032
33. E09000033

Crime
Column: BOCU
Unique values: 32
Values:
01. Barking and Dagenham
02. Barnet
03. Bexley
04. Brent
05. Bromley
06. Camden
07. Croydon
08. Ealing
09. Enfield
10. Greenwich
11. Hackney
12. Hammersmith and Fulham
13. Haringey
14. Harrow
15. Havering
16. Hillingdon
17. Hounslow
18. Islington
19. Kensington and Chelsea
20. Kingston upon Thames
21. Lambeth
22. Lewisham
23. Merton
24. Newham
25. Redbridge
26. Richmond upon Thames
27. Southwark
28. Sutton
29. Tower Hamlets
30. Waltham Fores

In [24]:
# Inspect Crime geographic identifiers

crime = datasets["Crime"]

crime_values = sorted(
    crime["BOCU"]
    .dropna()
    .astype(str)
    .unique()
)

print("=== CRIME BOCU IDENTIFIERS ===")
print("Unique BOCU values:", len(crime_values))
print()

for i, value in enumerate(crime_values, start=1):
    print(f"{i:02d}. {value}")

=== CRIME BOCU IDENTIFIERS ===
Unique BOCU values: 32

01. Barking and Dagenham
02. Barnet
03. Bexley
04. Brent
05. Bromley
06. Camden
07. Croydon
08. Ealing
09. Enfield
10. Greenwich
11. Hackney
12. Hammersmith and Fulham
13. Haringey
14. Harrow
15. Havering
16. Hillingdon
17. Hounslow
18. Islington
19. Kensington and Chelsea
20. Kingston upon Thames
21. Lambeth
22. Lewisham
23. Merton
24. Newham
25. Redbridge
26. Richmond upon Thames
27. Southwark
28. Sutton
29. Tower Hamlets
30. Waltham Forest
31. Wandsworth
32. Westminster


In [25]:
# Build canonical borough mapping for integration

borough_mapping = canonical_reference[
    [
        "Code",
        "Name",
        "Borough_Match_Key"
    ]
].copy()

borough_mapping = borough_mapping.rename(
    columns={
        "Code": "Borough_Code",
        "Name": "Borough_Name"
    }
)

print("=== CANONICAL BOROUGH MAPPING ===")
print("Shape:", borough_mapping.shape)
print("Unique Borough Codes:", borough_mapping["Borough_Code"].nunique())
print("Unique Match Keys:", borough_mapping["Borough_Match_Key"].nunique())
print("Duplicate Match Keys:", borough_mapping["Borough_Match_Key"].duplicated().sum())
print("Duplicate Borough Codes:", borough_mapping["Borough_Code"].duplicated().sum())

display(borough_mapping)

=== CANONICAL BOROUGH MAPPING ===
Shape: (33, 3)
Unique Borough Codes: 33
Unique Match Keys: 33
Duplicate Match Keys: 0
Duplicate Borough Codes: 0


,Borough_Code,Borough_Name,Borough_Match_Key
0,E09000001,City of London,CITY OF LONDON
1,E09000002,Barking and Dagenham,BARKING AND DAGENHAM
2,E09000003,Barnet,BARNET
3,E09000004,Bexley,BEXLEY
4,E09000005,Brent,BRENT
5,E09000006,Bromley,BROMLEY
6,E09000007,Camden,CAMDEN
7,E09000008,Croydon,CROYDON
8,E09000009,Ealing,EALING
9,E09000010,Enfield,ENFIELD


In [26]:
# Create integration copies of Property and Crime

property_integration = datasets["Property"].copy()
crime_integration = datasets["Crime"].copy()

# Create normalized matching keys

property_integration["Borough_Match_Key"] = (
    normalize_borough_for_integration(
        property_integration["District"]
    )
)

crime_integration["Borough_Match_Key"] = (
    normalize_borough_for_integration(
        crime_integration["BOCU"]
    )
)

print("=== INTEGRATION COPIES CREATED ===")

print(
    "Property:",
    property_integration.shape
)

print(
    "Crime:",
    crime_integration.shape
)

print(
    "Property original columns preserved:",
    set(datasets["Property"].columns).issubset(
        property_integration.columns
    )
)

print(
    "Crime original columns preserved:",
    set(datasets["Crime"].columns).issubset(
        crime_integration.columns
    )
)

=== INTEGRATION COPIES CREATED ===
Property: (264, 13)
Crime: (96, 16)
Property original columns preserved: True
Crime original columns preserved: True


In [27]:
print([
    name for name in globals()
    if "borough" in name.lower()
])

['borough_year_path', 'borough_year_features', 'property_boroughs', 'crime_boroughs', 'normalize_borough_name', 'borough_aliases', 'normalize_borough_for_integration', 'borough_mapping']


In [28]:
# Map Property and Crime layers to the canonical Borough_Code

# Property
property_integration["Borough_Match_Key"] = (
    property_integration["District"]
    .astype(str)
    .str.strip()
    .str.upper()
)

property_integration = property_integration.merge(
    borough_mapping[
        ["Borough_Code", "Borough_Match_Key"]
    ],
    on="Borough_Match_Key",
    how="left",
    validate="many_to_one"
)


# Crime
crime_integration["Borough_Match_Key"] = (
    crime_integration["BOCU"]
    .astype(str)
    .str.strip()
    .str.upper()
)

crime_integration = crime_integration.merge(
    borough_mapping[
        ["Borough_Code", "Borough_Match_Key"]
    ],
    on="Borough_Match_Key",
    how="left",
    validate="many_to_one"
)


# Audit
print("=== PROPERTY MAPPING AUDIT ===")
print("Rows:", len(property_integration))
print(
    "Missing Borough_Code:",
    property_integration["Borough_Code"].isna().sum()
)
print(
    "Unique Borough_Code:",
    property_integration["Borough_Code"].nunique()
)


print("\n=== CRIME MAPPING AUDIT ===")
print("Rows:", len(crime_integration))
print(
    "Missing Borough_Code:",
    crime_integration["Borough_Code"].isna().sum()
)
print(
    "Unique Borough_Code:",
    crime_integration["Borough_Code"].nunique()
)


print("\n=== CRIME BOROUGH COVERAGE ===")

crime_codes = set(
    crime_integration["Borough_Code"].dropna().unique()
)

canonical_codes = set(
    borough_mapping["Borough_Code"].unique()
)

print("Crime boroughs:", len(crime_codes))
print(
    "Missing from Crime:",
    sorted(canonical_codes - crime_codes)
)

=== PROPERTY MAPPING AUDIT ===
Rows: 264
Missing Borough_Code: 8
Unique Borough_Code: 32

=== CRIME MAPPING AUDIT ===
Rows: 96
Missing Borough_Code: 0
Unique Borough_Code: 32

=== CRIME BOROUGH COVERAGE ===
Crime boroughs: 32
Missing from Crime: ['E09000001']


In [29]:
print("=== PROPERTY UNMAPPED BOROUGHS ===")

unmapped_property = property_integration[
    property_integration["Borough_Code"].isna()
]

print("Number of unmapped rows:", len(unmapped_property))

print("\nDistrict values:")
print(
    unmapped_property["District"]
    .value_counts()
)

=== PROPERTY UNMAPPED BOROUGHS ===
Number of unmapped rows: 8

District values:
District
CITY OF WESTMINSTER    8
Name: count, dtype: int64


In [30]:
print("=== BOROUGH ALIASES ===")
print(borough_aliases)

=== BOROUGH ALIASES ===
{'CITY OF WESTMINSTER': 'WESTMINSTER'}


In [31]:
# Apply the existing borough alias to Property match keys

property_integration["Borough_Match_Key"] = (
    property_integration["District"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(borough_aliases)
)

print("=== PROPERTY MATCH KEY AUDIT ===")

unmapped_property = property_integration[
    property_integration["Borough_Code"].isna()
]

print(
    unmapped_property[
        ["District", "Borough_Match_Key"]
    ].drop_duplicates()
)

=== PROPERTY MATCH KEY AUDIT ===
               District Borough_Match_Key
56  CITY OF WESTMINSTER       WESTMINSTER


In [32]:
# Map Property to the canonical Borough_Code

property_integration = property_integration.drop(
    columns=["Borough_Code"],
)

property_integration = property_integration.merge(
    borough_mapping[
        ["Borough_Code", "Borough_Match_Key"]
    ],
    on="Borough_Match_Key",
    how="left",
    validate="many_to_one"
)

print("=== PROPERTY FINAL MAPPING AUDIT ===")
print("Rows:", len(property_integration))
print("Missing Borough_Code:", property_integration["Borough_Code"].isna().sum())
print("Unique Borough_Code:", property_integration["Borough_Code"].nunique())

=== PROPERTY FINAL MAPPING AUDIT ===
Rows: 264
Missing Borough_Code: 0
Unique Borough_Code: 33


In [33]:
# Verify Property grain after Borough mapping

property_duplicates = (
    property_integration
    .duplicated(subset=["District", "Year"], keep=False)
    .sum()
)

print("=== PROPERTY POST-MAPPING GRAIN AUDIT ===")
print("Rows:", len(property_integration))
print("Unique District-Year:", property_integration[["District", "Year"]].drop_duplicates().shape[0])
print("Duplicate District-Year rows:", property_duplicates)

=== PROPERTY POST-MAPPING GRAIN AUDIT ===
Rows: 264
Unique District-Year: 264
Duplicate District-Year rows: 0


In [34]:
# Map Crime to the canonical Borough_Code

crime_integration = crime_integration.drop(
    columns=["Borough_Code"]
)

crime_integration = crime_integration.merge(
    borough_mapping[
        ["Borough_Code", "Borough_Match_Key"]
    ],
    left_on="Merge_Key",
    right_on="Borough_Match_Key",
    how="left",
    validate="many_to_one"
)

print("=== CRIME FINAL MAPPING AUDIT ===")
print("Rows:", len(crime_integration))
print("Missing Borough_Code:", crime_integration["Borough_Code"].isna().sum())
print("Unique Borough_Code:", crime_integration["Borough_Code"].nunique())

=== CRIME FINAL MAPPING AUDIT ===
Rows: 96
Missing Borough_Code: 3
Unique Borough_Code: 31


In [35]:
# Inspect the unmapped Crime records

unmapped_crime = crime_integration[
    crime_integration["Borough_Code"].isna()
]

print("=== UNMAPPED CRIME RECORDS ===")
print("Number of unmapped rows:", len(unmapped_crime))

print("\nBOCU and Merge_Key:")
print(
    unmapped_crime[
        ["BOCU", "Merge_Key", "Year"]
    ].drop_duplicates()
)

=== UNMAPPED CRIME RECORDS ===
Number of unmapped rows: 3

BOCU and Merge_Key:
           BOCU            Merge_Key  Year
93  Westminster  CITY OF WESTMINSTER  2021
94  Westminster  CITY OF WESTMINSTER  2022
95  Westminster  CITY OF WESTMINSTER  2023


In [36]:
# Apply the existing borough aliases to Crime match keys

crime_integration["Borough_Match_Key"] = (
    crime_integration["Merge_Key"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(borough_aliases)
)

print("=== CRIME MATCH KEY AUDIT ===")

unmapped_crime = crime_integration[
    crime_integration["Borough_Code"].isna()
]

print(
    unmapped_crime[
        ["BOCU", "Merge_Key", "Borough_Match_Key", "Year"]
    ].drop_duplicates()
)

=== CRIME MATCH KEY AUDIT ===
           BOCU            Merge_Key Borough_Match_Key  Year
93  Westminster  CITY OF WESTMINSTER       WESTMINSTER  2021
94  Westminster  CITY OF WESTMINSTER       WESTMINSTER  2022
95  Westminster  CITY OF WESTMINSTER       WESTMINSTER  2023


In [37]:
# Map Crime to the canonical Borough_Code after alias resolution

crime_integration = crime_integration.drop(
    columns=["Borough_Code"]
)

crime_integration = crime_integration.merge(
    borough_mapping[
        ["Borough_Code", "Borough_Match_Key"]
    ],
    on="Borough_Match_Key",
    how="left",
    validate="many_to_one"
)

print("=== CRIME FINAL MAPPING AUDIT ===")
print("Rows:", len(crime_integration))
print("Missing Borough_Code:", crime_integration["Borough_Code"].isna().sum())
print("Unique Borough_Code:", crime_integration["Borough_Code"].nunique())

=== CRIME FINAL MAPPING AUDIT ===
Rows: 96
Missing Borough_Code: 0
Unique Borough_Code: 32


In [38]:
# Verify Crime grain after Borough mapping

crime_duplicates = (
    crime_integration
    .duplicated(
        subset=["BOCU", "Year"],
        keep=False
    )
    .sum()
)

print("=== CRIME POST-MAPPING GRAIN AUDIT ===")
print("Rows:", len(crime_integration))
print(
    "Unique BOCU-Year:",
    crime_integration[["BOCU", "Year"]]
    .drop_duplicates()
    .shape[0]
)
print("Duplicate BOCU-Year rows:", crime_duplicates)

=== CRIME POST-MAPPING GRAIN AUDIT ===
Rows: 96
Unique BOCU-Year: 96
Duplicate BOCU-Year rows: 0


In [39]:
# Verify Population grain before integration

population_duplicates = (
    datasets["Population"]
    .duplicated(
        subset=["Code", "Year"],
        keep=False
    )
    .sum()
)

print("=== POPULATION GRAIN AUDIT ===")
print("Rows:", len(datasets["Population"]))
print(
    "Unique Code-Year:",
    datasets["Population"][["Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)
print("Duplicate Code-Year rows:", population_duplicates)

=== POPULATION GRAIN AUDIT ===
Rows: 396
Unique Code-Year: 396
Duplicate Code-Year rows: 0


In [40]:
# Create an independent Population integration copy

population_integration = datasets["Population"].copy(deep=True)

print("=== POPULATION INTEGRATION COPY ===")
print("Shape:", population_integration.shape)
print(
    "Original columns preserved:",
    list(population_integration.columns)
    == list(datasets["Population"].columns)
)

=== POPULATION INTEGRATION COPY ===
Shape: (396, 5)
Original columns preserved: True


In [41]:
# Map Population to the canonical Borough_Code

population_integration["Borough_Code"] = population_integration["Code"]

print("=== POPULATION CODE MAPPING AUDIT ===")
print("Rows:", len(population_integration))
print(
    "Missing Borough_Code:",
    population_integration["Borough_Code"].isna().sum()
)
print(
    "Unique Borough_Code:",
    population_integration["Borough_Code"].nunique()
)
print(
    "Codes not in canonical reference:",
    sorted(
        set(population_integration["Borough_Code"])
        - set(borough_mapping["Borough_Code"])
    )
)

=== POPULATION CODE MAPPING AUDIT ===
Rows: 396
Missing Borough_Code: 0
Unique Borough_Code: 33
Codes not in canonical reference: []


In [42]:
# Verify Population grain after Borough mapping

population_duplicates = (
    population_integration
    .duplicated(
        subset=["Code", "Year"],
        keep=False
    )
    .sum()
)

print("=== POPULATION POST-MAPPING GRAIN AUDIT ===")
print("Rows:", len(population_integration))
print(
    "Unique Code-Year:",
    population_integration[["Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)
print("Duplicate Code-Year rows:", population_duplicates)

=== POPULATION POST-MAPPING GRAIN AUDIT ===
Rows: 396
Unique Code-Year: 396
Duplicate Code-Year rows: 0


In [43]:
# Create an independent Income integration copy

income_integration = datasets["Income"].copy(deep=True)

print("=== INCOME INTEGRATION COPY ===")
print("Shape:", income_integration.shape)
print(
    "Original columns preserved:",
    list(income_integration.columns)
    == list(datasets["Income"].columns)
)

=== INCOME INTEGRATION COPY ===
Shape: (198, 8)
Original columns preserved: True


In [44]:
# Map Income to the canonical Borough_Code

income_integration["Borough_Code"] = income_integration["Code"]

print("=== INCOME CODE MAPPING AUDIT ===")
print("Rows:", len(income_integration))
print(
    "Missing Borough_Code:",
    income_integration["Borough_Code"].isna().sum()
)
print(
    "Unique Borough_Code:",
    income_integration["Borough_Code"].nunique()
)
print(
    "Codes not in canonical reference:",
    sorted(
        set(income_integration["Borough_Code"])
        - set(borough_mapping["Borough_Code"])
    )
)

=== INCOME CODE MAPPING AUDIT ===
Rows: 198
Missing Borough_Code: 0
Unique Borough_Code: 33
Codes not in canonical reference: []


In [45]:
# Verify Income grain after Borough mapping

income_duplicates = (
    income_integration
    .duplicated(
        subset=["Code", "Year"],
        keep=False
    )
    .sum()
)

print("=== INCOME POST-MAPPING GRAIN AUDIT ===")
print("Rows:", len(income_integration))
print(
    "Unique Code-Year:",
    income_integration[["Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)
print("Duplicate Code-Year rows:", income_duplicates)

=== INCOME POST-MAPPING GRAIN AUDIT ===
Rows: 198
Unique Code-Year: 198
Duplicate Code-Year rows: 0


In [46]:
# Create an independent PTAL integration copy

ptal_integration = datasets["PTAL"].copy(deep=True)

print("=== PTAL INTEGRATION COPY ===")
print("Shape:", ptal_integration.shape)
print(
    "Original columns preserved:",
    list(ptal_integration.columns)
    == list(datasets["PTAL"].columns)
)

=== PTAL INTEGRATION COPY ===
Shape: (33, 4)
Original columns preserved: True


In [47]:
# Verify PTAL grain before integration

ptal_duplicates = (
    ptal_integration
    .duplicated(
        subset=["Borough Code"],
        keep=False
    )
    .sum()
)

print("=== PTAL GRAIN AUDIT ===")
print("Rows:", len(ptal_integration))
print(
    "Unique Borough Codes:",
    ptal_integration["Borough Code"].nunique()
)
print("Duplicate Borough Code rows:", ptal_duplicates)
print(
    "Missing Borough Code:",
    ptal_integration["Borough Code"].isna().sum()
)

=== PTAL GRAIN AUDIT ===
Rows: 33
Unique Borough Codes: 33
Duplicate Borough Code rows: 0
Missing Borough Code: 0


In [48]:
# Verify PTAL codes against the canonical Borough reference

ptal_codes = set(
    ptal_integration["Borough Code"].dropna()
)

canonical_codes = set(
    borough_mapping["Borough_Code"]
)

print("=== PTAL CODE COMPATIBILITY AUDIT ===")
print("PTAL unique codes:", len(ptal_codes))
print("Canonical unique codes:", len(canonical_codes))

print(
    "Codes only in PTAL:",
    sorted(ptal_codes - canonical_codes)
)

print(
    "Canonical codes missing from PTAL:",
    sorted(canonical_codes - ptal_codes)
)

=== PTAL CODE COMPATIBILITY AUDIT ===
PTAL unique codes: 33
Canonical unique codes: 33
Codes only in PTAL: []
Canonical codes missing from PTAL: []


In [49]:
# Create an independent IMD integration copy

imd_integration = datasets["IMD"].copy(deep=True)

print("=== IMD INTEGRATION COPY ===")
print("Shape:", imd_integration.shape)
print(
    "Original columns preserved:",
    list(imd_integration.columns)
    == list(datasets["IMD"].columns)
)
print(
    "Unique Borough Codes:",
    imd_integration["Borough_Code"].nunique()
)
print(
    "Unique Years:",
    imd_integration["Year"].nunique()
)

=== IMD INTEGRATION COPY ===
Shape: (33, 13)
Original columns preserved: True
Unique Borough Codes: 33
Unique Years: 1


## Build Integrated Dataset

The Property layer is used as the integration backbone.

All additional feature layers will be joined using the canonical `Borough_Code` and `Year` where applicable. Original source layers remain unchanged, and missing values will be preserved where a feature layer does not provide coverage for a given borough or year.

In [50]:
# Create the integration backbone from Property

integrated_df = property_integration.copy(deep=True)

print("=== INTEGRATION BACKBONE ===")
print("Shape:", integrated_df.shape)
print("Rows:", len(integrated_df))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== INTEGRATION BACKBONE ===
Shape: (264, 14)
Rows: 264
Unique Borough-Year: 264


In [51]:
# Merge Population features into the integration backbone

integrated_df = integrated_df.merge(
    population_integration,
    on=["Borough_Code", "Year"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_population")
)

print("=== PROPERTY + POPULATION MERGE ===")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== PROPERTY + POPULATION MERGE ===
Rows: 264
Columns: 18
Unique Borough-Year: 264


In [52]:
# Check Population coverage after merge

population_columns = [
    column
    for column in population_integration.columns
    if column not in ["Borough_Code", "Code", "Year"]
]

print("=== POPULATION COVERAGE AFTER MERGE ===")
print("Population feature columns:", population_columns)

print(
    "\nMissing values by year:"
)

print(
    integrated_df
    .groupby("Year")[population_columns]
    .apply(lambda x: x.isna().all())
)

=== POPULATION COVERAGE AFTER MERGE ===
Population feature columns: ['Name', 'Geography', 'Population']

Missing values by year:
       Name  Geography  Population
Year                              
2018  False      False       False
2019  False      False       False
2020  False      False       False
2021  False      False       False
2022  False      False       False
2023   True       True        True
2024   True       True        True
2025   True       True        True


In [53]:
# Merge Income features into the integrated dataset

integrated_df = integrated_df.merge(
    income_integration,
    on=["Borough_Code", "Year"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_income")
)

print("=== INTEGRATION + INCOME MERGE ===")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== INTEGRATION + INCOME MERGE ===
Rows: 264
Columns: 25
Unique Borough-Year: 264


In [54]:
# Check Income coverage after merge

income_feature_columns = [
    column
    for column in income_integration.columns
    if column not in ["Borough_Code", "Code", "Year", "Area"]
]

print("=== INCOME COVERAGE AFTER MERGE ===")
print("Income feature columns:", income_feature_columns)

print("\nYears with complete missingness:")
print(
    integrated_df
    .groupby("Year")[income_feature_columns]
    .apply(lambda x: x.isna().all())
)

=== INCOME COVERAGE AFTER MERGE ===
Income feature columns: ['Tax_Year', 'Mean_Income', 'Median_Income', 'Number_of_Individuals', 'Merge_Key']

Years with complete missingness:
      Tax_Year  Mean_Income  Median_Income  Number_of_Individuals  Merge_Key
Year                                                                        
2018     False        False          False                  False      False
2019     False        False          False                  False      False
2020     False        False          False                  False      False
2021     False        False          False                  False      False
2022     False        False          False                  False      False
2023     False        False          False                  False      False
2024      True         True           True                   True       True
2025      True         True           True                   True       True


In [55]:
# Merge Crime features into the integrated dataset

integrated_df = integrated_df.merge(
    crime_integration,
    on=["Borough_Code", "Year"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_crime")
)

print("=== INTEGRATION + CRIME MERGE ===")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== INTEGRATION + CRIME MERGE ===
Rows: 264
Columns: 42
Unique Borough-Year: 264


In [56]:
# Check Crime coverage after merge

crime_feature_columns = [
    column
    for column in crime_integration.columns
    if column not in [
        "Borough_Code",
        "BOCU",
        "Year",
        "Merge_Key"
    ]
]

print("=== CRIME COVERAGE AFTER MERGE ===")
print("Crime feature columns:", crime_feature_columns)

print("\nYears with complete missingness:")
print(
    integrated_df
    .groupby("Year")[crime_feature_columns]
    .apply(lambda x: x.isna().all())
)

print("\nCity of London Crime coverage:")
print(
    integrated_df[
        integrated_df["Borough_Code"] == "E09000001"
    ][
        ["Year"] + crime_feature_columns
    ]
)

=== CRIME COVERAGE AFTER MERGE ===
Crime feature columns: ['ARSON AND CRIMINAL DAMAGE', 'BURGLARY', 'DRUG OFFENCES', 'FRAUD AND FORGERY', 'MISCELLANEOUS CRIMES AGAINST SOCIETY', 'POSSESSION OF WEAPONS', 'PUBLIC ORDER OFFENCES', 'ROBBERY', 'SEXUAL OFFENCES', 'THEFT', 'VEHICLE OFFENCES', 'VIOLENCE AGAINST THE PERSON', 'Borough_Match_Key_x', 'Borough_Match_Key_y', 'Borough_Match_Key']

Years with complete missingness:
      ARSON AND CRIMINAL DAMAGE  BURGLARY  DRUG OFFENCES  FRAUD AND FORGERY  \
Year                                                                          
2018                       True      True           True               True   
2019                       True      True           True               True   
2020                       True      True           True               True   
2021                      False     False          False              False   
2022                      False     False          False              False   
2023                      Fa

In [57]:
print("=== CRIME FINAL COVERAGE CHECK ===")

crime_cols = [
    c for c in crime_integration.columns
    if c not in ["Borough_Code", "BOCU", "Year", "Merge_Key", "Borough_Match_Key"]
]

print("Crime feature count:", len(crime_cols))

for year in sorted(integrated_df["Year"].unique()):
    rows_with_crime = integrated_df.loc[
        integrated_df["Year"] == year,
        crime_cols
    ].notna().any(axis=1).sum()

    print(
        f"{year}: {rows_with_crime}/{len(integrated_df[integrated_df['Year'] == year])} "
        "boroughs with Crime data"
    )

city_london_rows = integrated_df[
    integrated_df["Borough_Code"] == "E09000001"
]

print(
    "City of London rows with Crime data:",
    city_london_rows[crime_cols].notna().any(axis=1).sum()
)

=== CRIME FINAL COVERAGE CHECK ===
Crime feature count: 14
2018: 0/33 boroughs with Crime data
2019: 0/33 boroughs with Crime data
2020: 0/33 boroughs with Crime data
2021: 32/33 boroughs with Crime data
2022: 32/33 boroughs with Crime data
2023: 32/33 boroughs with Crime data
2024: 0/33 boroughs with Crime data
2025: 0/33 boroughs with Crime data
City of London rows with Crime data: 0


In [58]:
print("=== PTAL COLUMNS FOR INTEGRATION ===")
print(ptal_integration.columns.tolist())

=== PTAL COLUMNS FOR INTEGRATION ===
['Borough Code', 'Borough Name', 'AvPTAI2015', 'PTAL']


In [59]:
# Merge PTAL borough-level features into the integrated dataset

ptal_for_merge = ptal_integration[
    ["Borough Code", "AvPTAI2015", "PTAL"]
].rename(
    columns={"Borough Code": "Borough_Code"}
)

integrated_df = integrated_df.merge(
    ptal_for_merge,
    on="Borough_Code",
    how="left",
    validate="many_to_one"
)

print("=== INTEGRATION + PTAL MERGE ===")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)
print(
    "Missing PTAL:",
    integrated_df["PTAL"].isna().sum()
)
print(
    "Missing AvPTAI2015:",
    integrated_df["AvPTAI2015"].isna().sum()
)

=== INTEGRATION + PTAL MERGE ===
Rows: 264
Columns: 44
Unique Borough-Year: 264
Missing PTAL: 0
Missing AvPTAI2015: 0


In [60]:
# Prepare IMD features for integration

imd_for_merge = imd_integration.copy(deep=True)

imd_for_merge = imd_for_merge.drop(
    columns=["Borough_Name"]
)

print("=== IMD READY FOR MERGE ===")
print("Shape:", imd_for_merge.shape)
print("Columns:", imd_for_merge.columns.tolist())

=== IMD READY FOR MERGE ===
Shape: (33, 12)
Columns: ['Borough_Code', 'IMD_Average_Rank', 'IMD_Rank_of_Average_Rank', 'IMD_Average_Score', 'IMD_Rank_of_Average_Score', 'IMD_Proportion_Most_Deprived_10pct', 'IMD_Rank_of_Proportion_Most_Deprived_10pct', 'IMD_Extent', 'IMD_Rank_of_Extent', 'IMD_Local_Concentration', 'IMD_Rank_of_Local_Concentration', 'Year']


In [61]:
# Merge IMD features into the integrated dataset

integrated_df = integrated_df.merge(
    imd_for_merge,
    on=["Borough_Code", "Year"],
    how="left",
    validate="one_to_one"
)

print("=== INTEGRATION + IMD MERGE ===")
print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== INTEGRATION + IMD MERGE ===
Rows: 264
Columns: 54
Unique Borough-Year: 264


In [62]:
# Check IMD coverage after merge

imd_feature_columns = [
    column
    for column in imd_for_merge.columns
    if column not in ["Borough_Code", "Year"]
]

print("=== IMD COVERAGE AFTER MERGE ===")

for year in sorted(integrated_df["Year"].unique()):
    rows_with_imd = integrated_df.loc[
        integrated_df["Year"] == year,
        imd_feature_columns
    ].notna().any(axis=1).sum()

    print(
        f"{year}: {rows_with_imd}/{len(integrated_df[integrated_df['Year'] == year])} "
        "boroughs with IMD data"
    )

=== IMD COVERAGE AFTER MERGE ===
2018: 0/33 boroughs with IMD data
2019: 33/33 boroughs with IMD data
2020: 0/33 boroughs with IMD data
2021: 0/33 boroughs with IMD data
2022: 0/33 boroughs with IMD data
2023: 0/33 boroughs with IMD data
2024: 0/33 boroughs with IMD data
2025: 0/33 boroughs with IMD data


In [63]:
print("=== FINAL INTEGRATED COLUMNS ===")

for i, column in enumerate(integrated_df.columns, start=1):
    print(f"{i:02d}. {column}")

=== FINAL INTEGRATED COLUMNS ===
01. District
02. Year
03. Transactions
04. Average_Price
05. Median_Price
06. Min_Price
07. Max_Price
08. Price_STD
09. Average_Price_Growth
10. Median_Price_Growth
11. Target_Average_Price_Growth
12. Target_Median_Price_Growth
13. Borough_Match_Key
14. Borough_Code
15. Code
16. Name
17. Geography
18. Population
19. Code_income
20. Area
21. Tax_Year
22. Mean_Income
23. Median_Income
24. Number_of_Individuals
25. Merge_Key
26. BOCU
27. ARSON AND CRIMINAL DAMAGE
28. BURGLARY
29. DRUG OFFENCES
30. FRAUD AND FORGERY
31. MISCELLANEOUS CRIMES AGAINST SOCIETY
32. POSSESSION OF WEAPONS
33. PUBLIC ORDER OFFENCES
34. ROBBERY
35. SEXUAL OFFENCES
36. THEFT
37. VEHICLE OFFENCES
38. VIOLENCE AGAINST THE PERSON
39. Merge_Key_crime
40. Borough_Match_Key_x
41. Borough_Match_Key_y
42. Borough_Match_Key_crime
43. AvPTAI2015
44. PTAL
45. IMD_Average_Rank
46. IMD_Rank_of_Average_Rank
47. IMD_Average_Score
48. IMD_Rank_of_Average_Score
49. IMD_Proportion_Most_Deprived_10pct


In [64]:
print("=== FINAL INTEGRATED COLUMNS ===")

columns = integrated_df.columns.tolist()

for start in range(0, len(columns), 10):
    print(
        f"\nColumns {start + 1}-{min(start + 10, len(columns))}:"
    )
    for i, column in enumerate(
        columns[start:start + 10],
        start=start + 1
    ):
        print(f"{i:02d}. {column}")

=== FINAL INTEGRATED COLUMNS ===

Columns 1-10:
01. District
02. Year
03. Transactions
04. Average_Price
05. Median_Price
06. Min_Price
07. Max_Price
08. Price_STD
09. Average_Price_Growth
10. Median_Price_Growth

Columns 11-20:
11. Target_Average_Price_Growth
12. Target_Median_Price_Growth
13. Borough_Match_Key
14. Borough_Code
15. Code
16. Name
17. Geography
18. Population
19. Code_income
20. Area

Columns 21-30:
21. Tax_Year
22. Mean_Income
23. Median_Income
24. Number_of_Individuals
25. Merge_Key
26. BOCU
27. ARSON AND CRIMINAL DAMAGE
28. BURGLARY
29. DRUG OFFENCES
30. FRAUD AND FORGERY

Columns 31-40:
31. MISCELLANEOUS CRIMES AGAINST SOCIETY
32. POSSESSION OF WEAPONS
33. PUBLIC ORDER OFFENCES
34. ROBBERY
35. SEXUAL OFFENCES
36. THEFT
37. VEHICLE OFFENCES
38. VIOLENCE AGAINST THE PERSON
39. Merge_Key_crime
40. Borough_Match_Key_x

Columns 41-50:
41. Borough_Match_Key_y
42. Borough_Match_Key_crime
43. AvPTAI2015
44. PTAL
45. IMD_Average_Rank
46. IMD_Rank_of_Average_Rank
47. IMD_Aver

In [65]:
# Classify final integrated columns by source layer

column_groups = {
    "Property": [
        "District", "Year", "Transactions",
        "Average_Price", "Median_Price",
        "Min_Price", "Max_Price", "Price_STD",
        "Average_Price_Growth", "Median_Price_Growth",
        "Target_Average_Price_Growth",
        "Target_Median_Price_Growth"
    ],

    "Population": [
        "Code", "Name", "Geography", "Population"
    ],

    "Income": [
        "Code_income", "Area", "Tax_Year",
        "Mean_Income", "Median_Income",
        "Number_of_Individuals", "Merge_Key"
    ],

    "Crime": [
        "BOCU", "Merge_Key"
    ],

    "PTAL": [
        "AvPTAI2015", "PTAL"
    ],

    "IMD": [
        "IMD_Average_Rank",
        "IMD_Rank_of_Average_Rank",
        "IMD_Average_Score",
        "IMD_Rank_of_Average_Score",
        "IMD_Proportion_Most_Deprived_10pct",
        "IMD_Rank_of_Proportion_Most_Deprived_10pct",
        "IMD_Extent",
        "IMD_Rank_of_Extent",
        "IMD_Local_Concentration",
        "IMD_Rank_of_Local_Concentration"
    ]
}

for group, columns in column_groups.items():
    existing = [c for c in columns if c in integrated_df.columns]

    print(f"\n=== {group} ===")
    print("Count:", len(existing))
    print(existing)


=== Property ===
Count: 12
['District', 'Year', 'Transactions', 'Average_Price', 'Median_Price', 'Min_Price', 'Max_Price', 'Price_STD', 'Average_Price_Growth', 'Median_Price_Growth', 'Target_Average_Price_Growth', 'Target_Median_Price_Growth']

=== Population ===
Count: 4
['Code', 'Name', 'Geography', 'Population']

=== Income ===
Count: 7
['Code_income', 'Area', 'Tax_Year', 'Mean_Income', 'Median_Income', 'Number_of_Individuals', 'Merge_Key']

=== Crime ===
Count: 2
['BOCU', 'Merge_Key']

=== PTAL ===
Count: 2
['AvPTAI2015', 'PTAL']

=== IMD ===
Count: 10
['IMD_Average_Rank', 'IMD_Rank_of_Average_Rank', 'IMD_Average_Score', 'IMD_Rank_of_Average_Score', 'IMD_Proportion_Most_Deprived_10pct', 'IMD_Rank_of_Proportion_Most_Deprived_10pct', 'IMD_Extent', 'IMD_Rank_of_Extent', 'IMD_Local_Concentration', 'IMD_Rank_of_Local_Concentration']


In [66]:
# Remove integration-only helper columns

columns_to_drop = [
    "Borough_Match_Key",
    "Code",
    "Name",
    "Geography",
    "Code_income",
    "Area",
    "Tax_Year",
    "Merge_Key",
    "BOCU"
]

integrated_df = integrated_df.drop(
    columns=columns_to_drop
)

print("=== POST-CLEANUP DATASET ===")
print("Shape:", integrated_df.shape)
print(
    "Unique Borough-Year:",
    integrated_df[["Borough_Code", "Year"]]
    .drop_duplicates()
    .shape[0]
)

=== POST-CLEANUP DATASET ===
Shape: (264, 45)
Unique Borough-Year: 264


In [67]:
# Final integration missing-value audit

print("=== FINAL INTEGRATION MISSINGNESS AUDIT ===")

missing_summary = (
    integrated_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

print(missing_summary)

=== FINAL INTEGRATION MISSINGNESS AUDIT ===
IMD_Extent                                    231
IMD_Rank_of_Proportion_Most_Deprived_10pct    231
IMD_Proportion_Most_Deprived_10pct            231
IMD_Average_Rank                              231
IMD_Rank_of_Average_Rank                      231
IMD_Local_Concentration                       231
IMD_Rank_of_Local_Concentration               231
IMD_Rank_of_Extent                            231
IMD_Average_Score                             231
IMD_Rank_of_Average_Score                     231
Borough_Match_Key_y                           171
FRAUD AND FORGERY                             168
ROBBERY                                       168
MISCELLANEOUS CRIMES AGAINST SOCIETY          168
POSSESSION OF WEAPONS                         168
PUBLIC ORDER OFFENCES                         168
Merge_Key_crime                               168
VIOLENCE AGAINST THE PERSON                   168
VEHICLE OFFENCES                              168
THEFT 

In [68]:
# Remove remaining integration helper columns

helper_columns = [
    "Borough_Match_Key_x",
    "Borough_Match_Key_y",
    "Borough_Match_Key_crime",
    "Merge_Key_crime"
]

integrated_df = integrated_df.drop(
    columns=helper_columns
)

print("=== HELPER COLUMN CLEANUP ===")
print("Shape:", integrated_df.shape)
print("Unique Borough-Year:",
      integrated_df[["Borough_Code", "Year"]]
      .drop_duplicates()
      .shape[0])

=== HELPER COLUMN CLEANUP ===
Shape: (264, 41)
Unique Borough-Year: 264


## Final Integration

The Property dataset provides the core **borough-year backbone** of the integrated dataset. Population, income, crime, PTAL, and IMD data were mapped to the canonical `Borough_Code` and integrated using appropriate geographic and temporal keys.

### Integration principles

* `Borough_Code` is the canonical geographic identifier across datasets.
* `Year` is used as the temporal key where the source data is year-specific.
* Borough-level datasets without a yearly dimension, such as PTAL, are treated as static borough-level features.
* IMD is available for 2019 only and is therefore matched only to the 2019 observations.
* Source datasets with limited temporal coverage are not artificially extended or imputed during integration.
* The final grain is **one row per Borough-Year**.

### Final dataset

The integrated dataset contains **264 Borough-Year observations**, representing 33 London boroughs across the Property data period.

Missing values in Population, Income, Crime, IMD, and target variables reflect the actual temporal coverage or modelling structure of the source data and are handled separately during subsequent analysis and feature engineering.

The integration stage is now complete.


In [69]:
print("=== FINAL DATASET SANITY CHECK ===")

print("Shape:", integrated_df.shape)

print("\nData types:")
print(integrated_df.dtypes)

print("\nDuplicate rows:", integrated_df.duplicated().sum())

print("\nDuplicate Borough-Year:",
      integrated_df.duplicated(
          subset=["Borough_Code", "Year"]
      ).sum())

=== FINAL DATASET SANITY CHECK ===
Shape: (264, 41)

Data types:
District                                       object
Year                                            int64
Transactions                                    int64
Average_Price                                 float64
Median_Price                                  float64
Min_Price                                       int64
Max_Price                                       int64
Price_STD                                     float64
Average_Price_Growth                          float64
Median_Price_Growth                           float64
Target_Average_Price_Growth                   float64
Target_Median_Price_Growth                    float64
Borough_Code                                   object
Population                                    float64
Mean_Income                                   float64
Median_Income                                 float64
Number_of_Individuals                         float64
ARSON AND CRIMINA

## Integration Complete

The integration stage is complete.

The final dataset contains **264 unique Borough-Year observations**, representing **33 London boroughs across 2018–2025**. Property data forms the core borough-year backbone, with Population, Income, Crime, PTAL, and IMD features integrated using the appropriate geographic and temporal keys.

Structural checks confirm:

* No duplicate rows
* No duplicate `Borough_Code`–`Year` combinations
* No loss or duplication of observations during integration

Missing values are retained where they reflect the original temporal coverage of the source datasets. They will be investigated and handled during the subsequent **Data Quality and Feature Analysis** stage.

The integrated dataset is now ready for quality assessment and feature analysis.


In [70]:
# Export the integrated borough-year dataset

integration_output_path = (
    "../data/processed/"
    "london_property_intelligence_integrated_borough_year.parquet"
)

integrated_df.to_parquet(
    integration_output_path,
    index=False
)

print("=== INTEGRATION DATASET EXPORTED ===")
print("Path:", integration_output_path)
print("Shape:", integrated_df.shape)

=== INTEGRATION DATASET EXPORTED ===
Path: ../data/processed/london_property_intelligence_integrated_borough_year.parquet
Shape: (264, 41)


In [71]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
